# 🏥 Health Deterioration Prediction — Lighthouse Residents

This notebook predicts which residents are at risk of **health deterioration** based on:
- Trends in health & wellbeing scores over time
- Incident history (severity, frequency, type)
- Current risk level and case category
- Intervention plan status

**Output:** A risk-ranked table of residents with predicted deterioration probability and key risk drivers.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

# Styling
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style='whitegrid', palette='muted')

DATA_DIR = '../lighthouse_csv_v7/lighthouse_csv_v7/'
print('Libraries loaded ✓')

## 2. Load Data

In [ ]:
health    = pd.read_csv(DATA_DIR + 'health_wellbeing_records.csv', parse_dates=['record_date'])
residents = pd.read_csv(DATA_DIR + 'residents.csv')
incidents = pd.read_csv(DATA_DIR + 'incident_reports.csv', parse_dates=['incident_date'])
plans     = pd.read_csv(DATA_DIR + 'intervention_plans.csv')

print(f'Health records : {len(health):,} rows across {health.resident_id.nunique()} residents')
print(f'Residents      : {len(residents):,} rows')
print(f'Incidents      : {len(incidents):,} rows')
print(f'Intervention   : {len(plans):,} rows')
health.head(3)

## 3. Feature Engineering

We build one row per resident by extracting signal from their health time-series, incidents, and plans.

In [ ]:
# ── 3a. Health score trends ────────────────────────────────────────────────
score_cols = ['general_health_score', 'nutrition_score', 'sleep_quality_score', 'energy_level_score']

def slope(series):
    """Linear regression slope using pure NumPy (negative = deteriorating)"""
    vals = series.values
    n = len(vals)
    if n < 2:
        return 0.0
    x = np.arange(n, dtype=float)
    x_mean, y_mean = x.mean(), vals.mean()
    num = ((x - x_mean) * (vals - y_mean)).sum()
    den = ((x - x_mean) ** 2).sum()
    return float(num / den) if den != 0 else 0.0

health_sorted = health.sort_values(['resident_id', 'record_date'])

health_features = health_sorted.groupby('resident_id').agg(
    # Latest scores
    latest_general_health   = ('general_health_score',  'last'),
    latest_nutrition        = ('nutrition_score',        'last'),
    latest_sleep            = ('sleep_quality_score',    'last'),
    latest_energy           = ('energy_level_score',     'last'),
    latest_bmi              = ('bmi',                    'last'),
    # Means
    mean_general_health     = ('general_health_score',  'mean'),
    mean_nutrition          = ('nutrition_score',        'mean'),
    mean_sleep              = ('sleep_quality_score',    'mean'),
    mean_energy             = ('energy_level_score',     'mean'),
    # Variability (std)
    std_general_health      = ('general_health_score',  'std'),
    std_energy              = ('energy_level_score',     'std'),
    # Number of records
    n_health_records        = ('health_record_id',       'count'),
    # Checkup compliance
    pct_medical_checkup     = ('medical_checkup_done',   'mean'),
    pct_psych_checkup       = ('psychological_checkup_done', 'mean'),
).reset_index()

# Add slopes (trend) per resident per score
for col in score_cols:
    slopes = health_sorted.groupby('resident_id')[col].apply(slope).reset_index()
    slopes.columns = ['resident_id', f'slope_{col}']
    health_features = health_features.merge(slopes, on='resident_id', how='left')

# Composite: average of the four latest scores
health_features['composite_latest'] = health_features[[
    'latest_general_health','latest_nutrition','latest_sleep','latest_energy'
]].mean(axis=1)

# Composite trend (average of 4 slopes)
health_features['composite_slope'] = health_features[[
    'slope_general_health_score','slope_nutrition_score',
    'slope_sleep_quality_score','slope_energy_level_score'
]].mean(axis=1)

print(f'Health features shape: {health_features.shape}')
health_features[['resident_id','composite_latest','composite_slope']].head()

In [ ]:
# ── 3b. Incident features ─────────────────────────────────────────────────
severity_map  = {'Low': 1, 'Medium': 2, 'High': 3}
incidents['severity_num'] = incidents['severity'].map(severity_map)

inc_features = incidents.groupby('resident_id').agg(
    total_incidents         = ('incident_id',    'count'),
    high_severity_incidents = ('severity_num',   lambda x: (x == 3).sum()),
    mean_severity           = ('severity_num',   'mean'),
    n_selfharm              = ('incident_type',  lambda x: (x == 'SelfHarm').sum()),
    n_runaway               = ('incident_type',  lambda x: (x == 'RunawayAttempt').sum()),
    n_medical               = ('incident_type',  lambda x: (x == 'Medical').sum()),
    pct_unresolved          = ('resolved',        lambda x: (~x).mean()),
    follow_up_needed        = ('follow_up_required', 'sum'),
).reset_index()

print(f'Incident features shape: {inc_features.shape}')
inc_features.head(3)

In [ ]:
# ── 3c. Intervention plan features ────────────────────────────────────────
plan_features = plans.groupby('resident_id').agg(
    total_plans             = ('plan_id',       'count'),
    n_completed_plans       = ('status',        lambda x: (x == 'Completed').sum()),
    n_active_plans          = ('status',        lambda x: (x == 'Active').sum()),
).reset_index()
plan_features['plan_completion_rate'] = (
    plan_features['n_completed_plans'] / plan_features['total_plans'].replace(0, np.nan)
)

print(f'Plan features shape: {plan_features.shape}')

In [ ]:
# ── 3d. Resident base features ────────────────────────────────────────────
risk_order = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}

# Pick vulnerability sub-category (prefer sub_cat_at_risk, fallback gracefully)
vuln_col = 'sub_cat_at_risk' if 'sub_cat_at_risk' in residents.columns else residents.columns[residents.columns.str.startswith('sub_cat_')][0]

res = residents[['resident_id','case_category','current_risk_level','initial_risk_level',
                 'sub_cat_physical_abuse','sub_cat_sexual_abuse', vuln_col,
                 'is_pwd','has_special_needs','family_solo_parent']].copy()

res.columns = ['resident_id','case_category','current_risk_level','initial_risk_level',
               'sub_cat_physical_abuse','sub_cat_sexual_abuse','sub_cat_vulnerable',
               'is_pwd','has_special_needs','family_solo_parent']

res['current_risk_num']  = res['current_risk_level'].map(risk_order)
res['initial_risk_num']  = res['initial_risk_level'].map(risk_order)
res['risk_escalated']    = (res['current_risk_num'] > res['initial_risk_num']).astype(int)
res['risk_improved']     = (res['current_risk_num'] < res['initial_risk_num']).astype(int)

# Encode case_category
le = LabelEncoder()
res['case_category_enc'] = le.fit_transform(res['case_category'].fillna('Unknown'))

print(f'Resident features: {res.shape}')
res[['resident_id','current_risk_level','risk_escalated','risk_improved']].head()

In [ ]:
# ── 3e. Merge all features ────────────────────────────────────────────────
df = res.merge(health_features, on='resident_id', how='left')
df = df.merge(inc_features,     on='resident_id', how='left')
df = df.merge(plan_features,    on='resident_id', how='left')

# Fill missing incident/plan counts with 0
incident_fill_cols = ['total_incidents','high_severity_incidents','mean_severity',
                      'n_selfharm','n_runaway','n_medical','pct_unresolved','follow_up_needed']
plan_fill_cols     = ['total_plans','n_completed_plans','n_active_plans','plan_completion_rate']
df[incident_fill_cols] = df[incident_fill_cols].fillna(0)
df[plan_fill_cols]     = df[plan_fill_cols].fillna(0)

print(f'Merged dataset: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

## 4. Define Target Variable

**Health Deterioration** = 1 if ANY of the following are true:
- Composite health trend slope < −0.002 (scores declining over time)
- Latest composite score < 2.9 (below a healthy baseline)
- Has ≥ 1 self-harm incident
- Current risk is High or Critical
- Risk escalated since admission

In [ ]:
df['deteriorating'] = (
    (df['composite_slope'] < -0.002) |
    (df['composite_latest'] < 2.9) |
    (df['n_selfharm'] >= 1) |
    (df['current_risk_num'] >= 2) |
    (df['risk_escalated'] == 1)
).astype(int)

print('Target distribution:')
print(df['deteriorating'].value_counts().rename({0: 'Stable', 1: 'Deteriorating'}))

fig, ax = plt.subplots(figsize=(5, 3))
counts = df['deteriorating'].value_counts().rename({0: 'Stable', 1: 'Deteriorating'})
colors = ['#4CAF50', '#F44336']
counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white', width=0.5)
ax.set_title('Health Deterioration — Target Distribution', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Number of Residents')
ax.set_xticklabels(counts.index, rotation=0)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width()/2, p.get_height() + 0.3), ha='center')
plt.tight_layout()
plt.show()

## 5. Train Prediction Model

In [ ]:
feature_cols = [
    # Health scores
    'composite_latest', 'composite_slope',
    'latest_general_health', 'latest_nutrition', 'latest_sleep', 'latest_energy',
    'std_general_health', 'std_energy',
    'latest_bmi', 'n_health_records',
    'pct_medical_checkup', 'pct_psych_checkup',
    'slope_general_health_score', 'slope_nutrition_score',
    'slope_sleep_quality_score', 'slope_energy_level_score',
    # Incidents
    'total_incidents', 'high_severity_incidents', 'mean_severity',
    'n_selfharm', 'n_runaway', 'n_medical', 'pct_unresolved', 'follow_up_needed',
    # Plans
    'plan_completion_rate', 'n_active_plans',
    # Resident context
    'initial_risk_num', 'case_category_enc',
    'sub_cat_physical_abuse', 'sub_cat_sexual_abuse', 'sub_cat_vulnerable',
    'is_pwd', 'has_special_needs',
]

X = df[feature_cols].copy()
y = df['deteriorating']

# Cast boolean columns to int
for col in X.select_dtypes(include='bool').columns:
    X[col] = X[col].astype(int)

print(f'Features: {X.shape[1]} | Samples: {X.shape[0]}')
print(f'Class balance: {y.mean():.1%} deteriorating')

In [ ]:
# ── Train a Random Forest with imputation pipeline ─────────────────────────
clf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=42
    ))
])

# Cross-validation (since n is small, use leave-one-out via StratifiedKFold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf_pipeline, X, y, cv=cv, scoring='roc_auc')
print(f'Cross-validated ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

# Fit on full data for prediction
clf_pipeline.fit(X, y)
df['deterioration_prob'] = clf_pipeline.predict_proba(X)[:, 1]
df['predicted_label']    = clf_pipeline.predict(X)

print('\nClassification report (in-sample):')
print(classification_report(y, df['predicted_label'], target_names=['Stable', 'Deteriorating']))

## 6. Results & Risk Rankings

In [ ]:
# ── Risk tier bucketing ────────────────────────────────────────────────────
def risk_tier(prob):
    if prob >= 0.75:   return 'Critical'
    elif prob >= 0.50: return 'High'
    elif prob >= 0.25: return 'Moderate'
    else:              return 'Low'

df['predicted_risk_tier'] = df['deterioration_prob'].apply(risk_tier)

results = df[[
    'resident_id', 'current_risk_level', 'case_category',
    'composite_latest', 'composite_slope',
    'total_incidents', 'n_selfharm', 'high_severity_incidents',
    'plan_completion_rate',
    'deterioration_prob', 'predicted_risk_tier'
]].sort_values('deterioration_prob', ascending=False).reset_index(drop=True)

results['composite_slope'] = results['composite_slope'].round(4)
results['composite_latest'] = results['composite_latest'].round(2)
results['deterioration_prob'] = results['deterioration_prob'].round(3)
results['plan_completion_rate'] = results['plan_completion_rate'].round(2)

print('=== TOP 10 HIGHEST RISK RESIDENTS ===')
print(results.head(10).to_string(index=True))

In [ ]:
# ── Summary table by risk tier ────────────────────────────────────────────
tier_summary = df.groupby('predicted_risk_tier').agg(
    n_residents          = ('resident_id',        'count'),
    avg_health_score     = ('composite_latest',   'mean'),
    avg_incidents        = ('total_incidents',    'mean'),
    avg_selfharm         = ('n_selfharm',         'mean'),
    avg_plan_completion  = ('plan_completion_rate','mean'),
).round(2)

tier_order = ['Critical', 'High', 'Moderate', 'Low']
tier_summary = tier_summary.reindex([t for t in tier_order if t in tier_summary.index])
print('\n=== SUMMARY BY PREDICTED RISK TIER ===')
print(tier_summary.to_string())

## 7. Visualizations

In [ ]:
# ── Plot 1: Deterioration probability by resident ─────────────────────────
tier_colors = {'Critical': '#c0392b', 'High': '#e67e22', 'Moderate': '#f1c40f', 'Low': '#27ae60'}

fig, ax = plt.subplots(figsize=(14, 5))
bar_colors = results['predicted_risk_tier'].map(tier_colors)
ax.bar(results.index, results['deterioration_prob'], color=bar_colors, edgecolor='white', linewidth=0.5)
ax.axhline(0.75, color='#c0392b', linestyle='--', linewidth=1)
ax.axhline(0.50, color='#e67e22', linestyle='--', linewidth=1)
ax.axhline(0.25, color='#f1c40f', linestyle='--', linewidth=1)
ax.set_xlabel('Resident (ranked by risk)', fontsize=11)
ax.set_ylabel('Deterioration Probability', fontsize=11)
ax.set_title('Health Deterioration Risk — All Residents', fontsize=13, fontweight='bold')
ax.set_xticks(results.index)
ax.set_xticklabels([f'R{int(r)}' for r in results['resident_id']], rotation=90, fontsize=7)
ax.set_ylim(0, 1.05)

legend_elements = [Patch(facecolor=c, label=t) for t, c in tier_colors.items()]
ax.legend(handles=legend_elements, loc='upper right', title='Risk Tier')
plt.tight_layout()
plt.savefig('risk_bar_chart.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Plot 2: Health score trend for top 5 highest-risk residents ───────────
top5_ids = results.head(5)['resident_id'].tolist()

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
for ax, rid in zip(axes, top5_ids):
    h = health_sorted[health_sorted['resident_id'] == rid].copy()
    h['composite'] = h[score_cols].mean(axis=1)
    ax.plot(h['record_date'], h['composite'], marker='o', markersize=4,
            color='#e74c3c', linewidth=2)
    ax.set_title(f'Resident {int(rid)}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Date', fontsize=8)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=8)
    tier = results[results['resident_id'] == rid]['predicted_risk_tier'].values[0]
    ax.set_facecolor({'Critical': '#fdecea', 'High': '#fef3e2', 'Moderate': '#fefde7', 'Low': '#eafaf1'}.get(tier, 'white'))

axes[0].set_ylabel('Composite Health Score', fontsize=10)
fig.suptitle('Health Score Trends — Top 5 Highest-Risk Residents', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('top5_trends.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Plot 3: Feature importances ───────────────────────────────────────────
model = clf_pipeline.named_steps['model']
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
top_n = importances.tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
colors_fi = ['#c0392b' if v > top_n.quantile(0.8) else '#3498db' for v in top_n.values]
top_n.plot(kind='barh', ax=ax, color=colors_fi, edgecolor='white')
ax.set_title('Top 15 Predictors of Health Deterioration', fontsize=12, fontweight='bold')
ax.set_xlabel('Feature Importance', fontsize=10)
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Plot 4: Risk tier donut chart ─────────────────────────────────────────
tier_counts = df['predicted_risk_tier'].value_counts().reindex(tier_order).dropna()
colors_donut = [tier_colors[t] for t in tier_counts.index]

fig, ax = plt.subplots(figsize=(5, 5))
wedges, texts, autotexts = ax.pie(
    tier_counts,
    labels=tier_counts.index,
    colors=colors_donut,
    autopct='%1.0f%%',
    startangle=90,
    wedgeprops=dict(width=0.55),
    textprops=dict(fontsize=11)
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')
ax.set_title('Residents by Predicted\nRisk Tier', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('risk_donut.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── Plot 5: Health score vs incidents scatter ─────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
scatter_colors = df['predicted_risk_tier'].map(tier_colors)
sc = ax.scatter(
    df['composite_latest'],
    df['total_incidents'],
    c=scatter_colors,
    s=80,
    alpha=0.8,
    edgecolors='white',
    linewidth=0.5
)
# Annotate high-risk points
for _, row in df[df['predicted_risk_tier'].isin(['Critical','High'])].iterrows():
    ax.annotate(f'R{int(row["resident_id"])}',
                (row['composite_latest'], row['total_incidents']),
                textcoords='offset points', xytext=(5, 2), fontsize=8, color='#555')

legend_handles = [Patch(facecolor=c, label=t) for t, c in tier_colors.items()]
ax.legend(handles=legend_handles, title='Predicted Risk', fontsize=9)
ax.set_xlabel('Composite Health Score (latest)', fontsize=10)
ax.set_ylabel('Total Incidents', fontsize=10)
ax.set_title('Health Score vs. Incident Frequency\nby Predicted Risk Tier', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('scatter_health_incidents.png', bbox_inches='tight', dpi=150)
plt.show()

## 8. Export Results

In [ ]:
# Export ranked risk table to CSV
export_cols = [
    'resident_id', 'current_risk_level', 'case_category',
    'composite_latest', 'composite_slope',
    'total_incidents', 'n_selfharm', 'high_severity_incidents',
    'plan_completion_rate', 'deterioration_prob', 'predicted_risk_tier'
]
results[export_cols].to_csv('health_deterioration_predictions.csv', index=False)
print('Saved: health_deterioration_predictions.csv')

print('\n=== FINAL RISK SUMMARY ===')
for tier in tier_order:
    subset = results[results['predicted_risk_tier'] == tier]
    if len(subset) > 0:
        ids = ', '.join([f'R{int(x)}' for x in subset['resident_id'].tolist()])
        print(f'{tier:10s} ({len(subset)} residents): {ids}')

## 9. Actionable Recommendations

| Risk Tier | Action |
|-----------|--------|
| **Critical** | Immediate case conference — escalate to medical/psychological review within 48 hours |
| **High** | Weekly social worker check-in — review and update intervention plan |
| **Moderate** | Bi-weekly monitoring — ensure checkup compliance and follow up on any open incidents |
| **Low** | Monthly review — continue current care plan |

> **Key risk drivers identified by the model:** declining health score trend, self-harm incidents, unresolved high-severity incidents, and low plan completion rate. Prioritize psychological check-ups and intervention plan adherence for residents in the top tiers.